<a href="https://colab.research.google.com/github/catch-twenty2/AstroChart_Analysis/blob/main/AstroChartAnalysis6light.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#First time before running the code you need to install these libraries
!pip install flatlib pyswisseph timezonefinder pytz ipywidgets geopy

click on the Play icon below on the left to run this code and scroll down to get the prompt for your AI assistant.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from datetime import datetime, timedelta
from timezonefinder import TimezoneFinder
import pytz
import swisseph as swe
import math
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderUnavailable
import traceback

# Define rulership for each zodiac sign (Modern Rulers)
planetary_rulers = {
    'Aries': 'Mars', 'Taurus': 'Venus', 'Gemini': 'Mercury', 'Cancer': 'Moon',
    'Leo': 'Sun', 'Virgo': 'Mercury', 'Libra': 'Venus', 'Scorpio': 'Pluto',
    'Sagittarius': 'Jupiter', 'Capricorn': 'Saturn', 'Aquarius': 'Uranus',
    'Pisces': 'Neptune'
}

# Helper function to determine the sign of a planet's longitude
def get_sign(longitude):
    signs = ['Aries', 'Taurus', 'Gemini', 'Cancer', 'Leo', 'Virgo',
             'Libra', 'Scorpio', 'Sagittarius', 'Capricorn', 'Aquarius', 'Pisces']
    lon = longitude % 360 # Ensure longitude is within 0-360 range
    return signs[int(lon / 30)]

# Helper function to find the planetary ruler of a sign
def get_ruler(sign):
    return planetary_rulers.get(sign, 'Unknown') # Use .get for safety

# Helper function to determine the house of a planet based on longitude using house cusps
def get_house_for_swisseph_planets(longitude, house_cusps):
    lon = longitude % 360 # Ensure longitude is within 0-360 range
    num_cusps = len(house_cusps)
    for i in range(1, num_cusps + 1):
        cusp_start = house_cusps[i - 1]; cusp_end = house_cusps[i % num_cusps]
        if cusp_end < cusp_start: # Handle wrap-around 0 degrees Aries
            if lon >= cusp_start or lon < cusp_end: return i
        else: # Normal case
            if cusp_start <= lon < cusp_end: return i
    return num_cusps if num_cusps > 0 else 12 # Default to last house

# Calculate the difference between two angles (longitudes) in a circular system (360 degrees)
def angular_distance(lon1, lon2):
    diff = abs(lon1 - lon2); return min(diff, 360 - diff)

# Helper function to check for aspects
def get_aspect(angle):
    aspects = {'Conjunction': 0, 'Opposition': 180, 'Trine': 120,'Square': 90, 'Sextile': 60, 'Quincunx': 150}
    orbs = {'Conjunction': 8, 'Opposition': 8, 'Trine': 8,'Square': 8, 'Sextile': 6, 'Quincunx': 3}
    min_diff = float('inf'); found_aspect = None
    for aspect, base_angle in aspects.items():
        diff = angular_distance(angle, base_angle)
        orb_allowed = orbs.get(aspect, 0)
        if diff <= orb_allowed:
             if diff < min_diff: min_diff = diff; found_aspect = aspect
    if found_aspect: return found_aspect, min_diff
    return None, None

# Function to calculate and display natal chart aspects
def display_natal_aspects(natal_chart_positions):
    print("\n--- Natal Aspects ---")
    planet_names = list(natal_chart_positions.keys()); aspect_found = False
    for i in range(len(planet_names)):
        for j in range(i + 1, len(planet_names)):
            planet1 = planet_names[i]; planet2 = planet_names[j]
            if {planet1, planet2} == {'North Node', 'South Node'}: continue # Skip NN-SN opposition
            lon1 = natal_chart_positions[planet1]['longitude']; lon2 = natal_chart_positions[planet2]['longitude']
            angle = angular_distance(lon1, lon2); aspect, orb = get_aspect(angle)
            if aspect: print(f"{planet1} {aspect} {planet2} (orb: {orb:.2f}°)"); aspect_found = True
    if not aspect_found: print("No major aspects found between calculated points.")

# Function to calculate and display transit aspects
def display_transit_aspects(natal_chart_positions, transit_positions):
    print("\n--- Transit Aspects (for Target Date) ---")
    aspect_found = False
    for transit_planet, transit_data in transit_positions.items():
        for natal_planet, natal_data in natal_chart_positions.items():
            transit_longitude = transit_data['longitude']; natal_longitude = natal_data['longitude']
            angle = angular_distance(transit_longitude, natal_longitude); aspect, orb = get_aspect(angle)
            if aspect: print(f"Transit {transit_planet} {aspect} Natal {natal_planet} (orb: {orb:.2f}°)"); aspect_found = True
    if not aspect_found: print("No major transit aspects found for the target date.")

# Function to get city and country from latitude and longitude
def get_city_country(latitude, longitude):
    try:
        geolocator = Nominatim(user_agent="astro_app_v6"); location = geolocator.reverse((latitude, longitude), exactly_one=True, timeout=10, language='en')
        if location and location.raw.get('address'): address = location.raw['address']; city = address.get('city', address.get('town', address.get('village', address.get('county','Unknown Locality')))); country = address.get('country', 'Unknown Country'); return city, country
        return "Unknown Locality", "Unknown Country"
    except (GeocoderTimedOut, GeocoderUnavailable) as e: print(f"Geocoding error: {e}. Returning Unknown."); return "Unknown Locality", "Unknown Country"
    except Exception as e: print(f"An unexpected geocoding error occurred: {e}"); return "Unknown Locality", "Unknown Country"

# Function to calculate UTC offset based on birth date and location
def calculate_utc_offset(year, month, day, hour, minute, latitude, longitude):
    tf = TimezoneFinder(); timezone_str = tf.timezone_at(lat=latitude, lng=longitude)
    if timezone_str is None: timezone_str = tf.closest_timezone_at(lat=latitude, lng=longitude);
    if timezone_str is None: raise ValueError("Could not determine the timezone for the given location.")
    try: local_timezone = pytz.timezone(timezone_str)
    except pytz.UnknownTimeZoneError: raise ValueError(f"Unknown timezone returned: {timezone_str}")
    try: naive_local_time = datetime(year, month, day, hour, minute)
    except ValueError as e_dt: raise ValueError(f"Invalid date/time input: {e_dt}")
    local_time = local_timezone.localize(naive_local_time, is_dst=None); utc_offset_timedelta = local_time.utcoffset()
    if utc_offset_timedelta is None: raise ValueError("Could not determine UTC offset after localization.")
    return utc_offset_timedelta.total_seconds() / 3600

# Sign properties dictionaries
sign_elements = {'Aries': 'Fire', 'Taurus': 'Earth', 'Gemini': 'Air', 'Cancer': 'Water','Leo': 'Fire', 'Virgo': 'Earth', 'Libra': 'Air', 'Scorpio': 'Water','Sagittarius': 'Fire', 'Capricorn': 'Earth', 'Aquarius': 'Air', 'Pisces': 'Water'}
sign_modes = {'Aries': 'Cardinal', 'Taurus': 'Fixed', 'Gemini': 'Mutable', 'Cancer': 'Cardinal','Leo': 'Fixed', 'Virgo': 'Mutable', 'Libra': 'Cardinal', 'Scorpio': 'Fixed','Sagittarius': 'Mutable', 'Capricorn': 'Cardinal', 'Aquarius': 'Fixed', 'Pisces': 'Mutable'}
sign_polarities = {'Aries': 'Masculine', 'Taurus': 'Feminine', 'Gemini': 'Masculine', 'Cancer': 'Feminine','Leo': 'Masculine', 'Virgo': 'Feminine', 'Libra': 'Masculine', 'Scorpio': 'Feminine','Sagittarius': 'Masculine', 'Capricorn': 'Feminine', 'Aquarius': 'Masculine', 'Pisces': 'Feminine'}

# Function to count elements, modes, and polarities
def count_elements_modes_polarities(natal_chart_positions):
    points_for_counts = list(natal_chart_positions.keys()) # Use all calculated points
    element_count = {'Fire': 0, 'Earth': 0, 'Air': 0, 'Water': 0}; mode_count = {'Cardinal': 0, 'Fixed': 0, 'Mutable': 0}; polarity_count = {'Masculine': 0, 'Feminine': 0}
    for planet in points_for_counts:
        data = natal_chart_positions.get(planet)
        if data: sign = data.get('sign')
        if sign: element, mode, polarity = sign_elements.get(sign), sign_modes.get(sign), sign_polarities.get(sign)
        if element: element_count[element] += 1
        if mode: mode_count[mode] += 1
        if polarity: polarity_count[polarity] += 1
    return element_count, mode_count, polarity_count

# Function to count hemisphere and quadrant balance
def count_hemisphere_quadrant_balance(natal_chart_positions):
    points_for_counts = list(natal_chart_positions.keys())
    hemisphere_count = {'Eastern': 0, 'Western': 0, 'Northern': 0, 'Southern': 0}; quadrant_count = {'First': 0, 'Second': 0, 'Third': 0, 'Fourth': 0}
    eastern_houses, western_houses = {1, 2, 3, 10, 11, 12}, {4, 5, 6, 7, 8, 9}; northern_houses, southern_houses = {1, 2, 3, 4, 5, 6}, {7, 8, 9, 10, 11, 12}
    quadrants = [{1, 2, 3}, {4, 5, 6}, {7, 8, 9}, {10, 11, 12}]; quad_keys = ['First', 'Second', 'Third', 'Fourth']
    for planet in points_for_counts:
         data = natal_chart_positions.get(planet)
         if data: house = data.get('house')
         if house:
            try: house_num = int(house)
            except ValueError: continue # Skip if house isn't integer
            if house_num in eastern_houses: hemisphere_count['Eastern'] += 1
            if house_num in western_houses: hemisphere_count['Western'] += 1
            if house_num in northern_houses: hemisphere_count['Northern'] += 1
            if house_num in southern_houses: hemisphere_count['Southern'] += 1
            for i, quad in enumerate(quadrants):
                 if house_num in quad: quadrant_count[quad_keys[i]] += 1; break
    return hemisphere_count, quadrant_count

# *** MODIFIED FUNCTION: Only calculates Lunar Nodes ***
def add_lunar_nodes(natal_chart_positions, jd_ut, house_cusps):
    # The True Node calculation does NOT require external ephemeris files
    try:
        # North Node
        data_nn = swe.calc_ut(jd_ut, swe.TRUE_NODE)
        lon_nn, lat_nn = data_nn[0][0], data_nn[0][1]
        sign_nn = get_sign(lon_nn)
        natal_chart_positions['North Node'] = {
            'longitude': lon_nn, 'latitude': lat_nn, 'sign': sign_nn,
            'house': get_house_for_swisseph_planets(lon_nn, house_cusps),
            'ruler': get_ruler(sign_nn)
        }
        # South Node (opposite)
        lon_sn = (lon_nn + 180) % 360
        sign_sn = get_sign(lon_sn)
        natal_chart_positions['South Node'] = {
            'longitude': lon_sn, 'latitude': -lat_nn, 'sign': sign_sn,
            'house': get_house_for_swisseph_planets(lon_sn, house_cusps),
            'ruler': get_ruler(sign_sn)
        }
    except Exception as e:
        print(f"Error calculating Lunar Nodes: {e}")

# List of fixed stars to include in the analysis
fixed_stars = ['Regulus', 'Sirius', 'Aldebaran', 'Spica', 'Antares', 'Fomalhaut', 'Vega']

# Calculate fixed star conjunctions
def calculate_fixed_star_conjunctions(natal_chart_positions, jd_ut):
    star_conjunctions = []
    orb_conjunct_star = 1.0 # Orb for fixed star conjunctions
    for star_name in fixed_stars:
        try:
            raw_result, star_name_out, err_msg = swe.fixstar_ut(star_name, jd_ut)
            if raw_result:
                star_longitude = raw_result[0]
                for planet, planet_data in natal_chart_positions.items():
                    planet_longitude = planet_data.get('longitude')
                    if planet_longitude is not None:
                        dist = angular_distance(planet_longitude, star_longitude)
                        if dist <= orb_conjunct_star:
                            star_conjunctions.append(f"{planet} conjunct {star_name} (Star Lon: {star_longitude:.2f}°, Orb: {dist:.2f}°)")
        except Exception as e:
            if "not found" not in str(e).lower():
                 print(f"Error processing star {star_name}: {e}")
    return star_conjunctions

# Function to display fixed star conjunctions
def display_fixed_star_conjunctions(fixed_star_conjunctions):
    print("\n--- Fixed Star Conjunctions ---")
    if fixed_star_conjunctions:
        for conjunction in fixed_star_conjunctions: print(conjunction)
    else: print("No significant fixed star conjunctions found (within 1° orb).")

# --- Widgets ---
user_name_input = widgets.Text(value='Neo', description="User Name:")
year_input = widgets.IntText(value=1981, description="Birth Year:")
month_input = widgets.IntText(value=10, description="Birth Month:")
day_input = widgets.IntText(value=10, description="Birth Day:")
time_of_birth_input = widgets.Text(value="18:19", description="Birth Time (HH:MM):")
latitude_input = widgets.FloatText(value=-13.5319, description="Latitude:")
longitude_input = widgets.FloatText(value=-71.9675, description="Longitude:")
target_year_input = widgets.IntText(value=2025, description="Target Year:")
target_month_input = widgets.IntText(value=12, description="Target Month:")
target_day_input = widgets.IntText(value=31, description="Target Day:")
button = widgets.Button(description="Calculate Chart")
output = widgets.Output()

# --- Main Calculation Function ---
def on_button_click(b):
    with output:
        clear_output()
        natal_chart_positions = {}
        try:
            # Capture and Validate Inputs
            user_name = user_name_input.value; year = year_input.value; month = month_input.value; day = day_input.value
            time_of_birth_str = time_of_birth_input.value; latitude = latitude_input.value; longitude = longitude_input.value
            target_year = target_year_input.value; target_month = target_month_input.value; target_day = target_day_input.value
            if not (1 <= month <= 12): raise ValueError("Month must be 1-12.")
            if not (-90 <= latitude <= 90): raise ValueError("Latitude must be -90 to 90.")
            if not (-180 <= longitude <= 180): raise ValueError("Longitude must be -180 to 180.")
            try: hour = int(time_of_birth_str[:2]); minute = int(time_of_birth_str[3:5])
            except (ValueError, IndexError): raise ValueError("Invalid time format HH:MM.")
            if not (0 <= hour <= 23 and 0 <= minute <= 59): raise ValueError("Invalid HH:MM range.")

            # Timezone and UTC Conversion
            utc_offset = calculate_utc_offset(year, month, day, hour, minute, latitude, longitude)
            local_birth_time = datetime(year, month, day, hour, minute)
            utc_birth_time = local_birth_time - timedelta(hours=utc_offset)

            # Geocoding
            city, country = get_city_country(latitude, longitude)

            # Julian Day for Natal Chart (UT)
            jd_ut = swe.julday(utc_birth_time.year, utc_birth_time.month, utc_birth_time.day,
                               utc_birth_time.hour + utc_birth_time.minute / 60.0)

            # Set Location
            swe.set_topo(longitude, latitude, 0)

            # Houses & AS/MC (Placidus)
            house_cusps, ascmc = swe.houses(jd_ut, latitude, longitude, b'P')

            # Natal Planets Calculation
            planets = {'Sun': swe.SUN, 'Moon': swe.MOON, 'Mercury': swe.MERCURY, 'Venus': swe.VENUS, 'Mars': swe.MARS, 'Jupiter': swe.JUPITER, 'Saturn': swe.SATURN, 'Uranus': swe.URANUS, 'Neptune': swe.NEPTUNE, 'Pluto': swe.PLUTO}
            for name, pid in planets.items():
                try: data = swe.calc_ut(jd_ut, pid)
                except Exception as e: print(f"Error calculating {name}: {e}"); continue
                if data: pos = data[0]; natal_chart_positions[name] = {'longitude': pos[0], 'latitude': pos[1], 'sign': get_sign(pos[0]), 'house': get_house_for_swisseph_planets(pos[0], house_cusps), 'ruler': get_ruler(get_sign(pos[0])), 'retrograde': pos[3] < 0}

            # Additional Points (Nodes only)
            add_lunar_nodes(natal_chart_positions, jd_ut, house_cusps)

            # Transit Calculation (Noon UT for target date)
            target_jd_ut = swe.julday(target_year, target_month, target_day, 12.0)
            transit_positions = {}
            for name, pid in planets.items(): # Transits for Sun-Pluto
                 try: data = swe.calc_ut(target_jd_ut, pid)
                 except Exception as e: print(f"Error calculating transit for {name}: {e}"); continue
                 if data: transit_positions[name] = {'longitude': data[0][0], 'sign': get_sign(data[0][0])}

            # --- Output Section ---
            print(f"\nDear Assistant, act as a professional Astrologer. Below is detailed astrological data for interpretation.\n")
            print(f"**Astrological Data:**")
            print(f"Name: {user_name}")
            print(f"Place of Birth: {city}, {country} ({latitude:.4f}, {longitude:.4f})")
            print(f"Date of Birth: {year}-{month:02d}-{day:02d}")
            print(f"Time of Birth: {time_of_birth_str} (Local)")
            print(f"Calculated UTC Offset: {utc_offset:+.2f} hours")
            print(f"UTC Time of Birth: {utc_birth_time.strftime('%Y-%m-%d %H:%M:%S')} UT")
            print(f"Target Date for Transits: {target_year}-{target_month:02d}-{target_day:02d}\n")
            print("--- Natal Chart Positions ---")
            display_order = ['Sun', 'Moon', 'Mercury', 'Venus', 'Mars', 'Jupiter', 'Saturn', 'Uranus', 'Neptune', 'Pluto','North Node', 'South Node']
            for planet in display_order:
                 data = natal_chart_positions.get(planet)
                 if data:
                     retro_marker = " (R)" if data.get('retrograde', False) else ""
                     lat_str = f", Lat: {data['latitude']:.2f}°"
                     deg_total = data['longitude']
                     deg_in_sign = int(deg_total % 30)
                     min_in_sign = int(((deg_total % 30) % 1) * 60)
                     pos_dms = f"{deg_in_sign:02d}°{min_in_sign:02d}'"
                     print(f"{planet}{retro_marker}: {pos_dms} {data['sign']} (Ruler: {data['ruler']}) in House {data['house']} [Lon: {deg_total:.2f}°{lat_str}]")
            # AS/MC
            ascendant_lon, mc_lon = ascmc[0], ascmc[1]
            asc_sign, mc_sign = get_sign(ascendant_lon), get_sign(mc_lon)
            asc_deg_in_sign, asc_min_in_sign = int(ascendant_lon % 30), int(((ascendant_lon % 30) % 1) * 60)
            mc_deg_in_sign, mc_min_in_sign = int(mc_lon % 30), int(((mc_lon % 30) % 1) * 60)
            print(f"\nAscendant (AC): {asc_deg_in_sign:02d}°{asc_min_in_sign:02d}' {asc_sign} ({ascendant_lon:.2f}°)")
            print(f"Midheaven (MC): {mc_deg_in_sign:02d}°{mc_min_in_sign:02d}' {mc_sign} ({mc_lon:.2f}°)")

            # Chart Analysis Counts
            print("\n--- Chart Analysis ---")
            element_count, mode_count, polarity_count = count_elements_modes_polarities(natal_chart_positions)
            hemisphere_count, quadrant_count = count_hemisphere_quadrant_balance(natal_chart_positions)
            print("Element Distribution:", ", ".join([f"{k}: {v}" for k,v in element_count.items()]))
            print("Mode Distribution:", ", ".join([f"{k}: {v}" for k,v in mode_count.items()]))
            print("Polarity Distribution:", ", ".join([f"{k}: {v}" for k,v in polarity_count.items()]))
            print("Hemisphere Balance:", ", ".join([f"{k}: {v}" for k,v in hemisphere_count.items()]))
            print("Quadrant Balance:", ", ".join([f"{k}: {v}" for k,v in quadrant_count.items()]))
            retrograde_planets = [p for p, d in natal_chart_positions.items() if d.get('retrograde', False)]
            print("\nRetrograde Planets:", ", ".join(retrograde_planets) if retrograde_planets else "No planets are retrograde.")


            # Fixed Stars Display (Declinations are not needed since we removed the function)
            fixed_star_conjunctions = calculate_fixed_star_conjunctions(natal_chart_positions, jd_ut)
            display_fixed_star_conjunctions(fixed_star_conjunctions)

            # Aspects Display
            display_natal_aspects(natal_chart_positions)
            display_transit_aspects(natal_chart_positions, transit_positions)

            # Interpretation Guidelines
            print(
                 "\n**Interpretation Guidelines:**\n"
                 "1. **Natal Chart Analysis:**\n"
                 "   - Examine the individual's personality traits, strengths, and potential challenges based on the Sun, Moon, and Ascendant.\n"
                 "   - Highlight important aspects between personal planets and outer planets to identify areas of potential growth or difficulty.\n\n"
                 "2. **Transit Analysis:**\n"
                 "   - Analyze how transiting planets interact with the natal chart, emphasizing long-term influences from outer planets.\n"
                 "   - Focus on aspects between natal planets and transiting planets that suggest turning points or opportunities.\n\n"
                 "3. **Key Themes and Life Cycles:**\n"
                 "   - Summarize life themes and cycles emerging from both the natal and transit charts.\n"
                 "   - Describe how these influences are likely to manifest in areas like career, relationships, or personal growth.\n\n"
                 "4. **Practical Guidance and Recommendations:**\n"
                 "   - Provide actionable advice on how to navigate any challenging transits.\n"
                 "   - Highlight opportunities for growth or positive change, and suggest areas for further exploration.\n\n"
                 "Finally, if there are any specific challenges or problematic aspects in the chart, recommend follow-up questions or next steps for further exploration.\n\nThank you!"
            )

        # Main Error Catching
        except Exception as e:
            print(f"An error occurred during chart calculation: {e}")
            print("--- Error Traceback ---"); print(traceback.format_exc()); print("--- End Traceback ---")

# --- Display Widgets and Assign Button Click ---
display(user_name_input, year_input, month_input, day_input, time_of_birth_input, latitude_input, longitude_input,
        target_year_input, target_month_input, target_day_input, button, output)
button.on_click(on_button_click)
